In [84]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
import asyncio
import uvicorn
import sqlite3

In [85]:
# Database initalization

# conn = sqlite3.connect("todo_db.db", check_same_thread=False)
# conn.row_factory = sqlite3.Row

def connect_db():
   conn = sqlite3.connect("todo_db.db")
   conn.row_factory = sqlite3.Row
   return conn

def convert_to_json(data):
  return [dict(res) for res in data]

def rows_to_dict(rows):
    return [dict(row) for row in rows]


In [86]:
def get_todos():
    conn = connect_db()
    rows = conn.execute("SELECT * FROM todo").fetchall()
    conn.close()
    return rows_to_dict(rows)

def get_todos_by_id(id):
    conn = connect_db()
    data = conn.execute(f"SELECT * FROM todo WHERE id = {id}").fetchone()
    conn.close()
    if data is None:
        raise HTTPException(status_code=404, detail="Todo not found.")
    return dict(data)

In [87]:
app = FastAPI()
templates = Jinja2Templates(directory="templates")

# Route (Maybe work with controllers)
@app.get("/", response_class=HTMLResponse)
def root():
  return """

  <html>
    <body>
      <a href="/docs">docs</a>
    </body>
  </html>

  """

@app.get("/todos", response_class=HTMLResponse)
def todos_html(request: Request):
  return templates.TemplateResponse(
    "todos.html",
    {
      "request": request,
      "todos": get_todos()
    }
  )

@app.get("/api/todos")
def todos():
  return {
    "data": get_todos()
  }

@app.get("/api/todos/{id}")
def todos_by_id(id):
  return {
    "data": get_todos_by_id(id)
  }

In [88]:
if __name__ == "__main__":
  config = uvicorn.Config(app)
  server = uvicorn.Server(config)
  await server.serve()

INFO:     Started server process [17012]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52188 - "GET /api/todos/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52188 - "GET /api/todos/1123 HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [17012]
